In [1]:
# ============================================================
#  Class Activity-6: Food Delivery Chatbot (ML)
#  BSSE23 & BSCE22 | SE305T & MD445T (Spring-26)
#  Dr. Muhammad Asif | ITU – Lahore
# ============================================================

# ── Task 1: Install / Import Libraries ──────────────────────
# In Google Colab run:
#   !pip install nltk scikit-learn numpy

import re
import random
import numpy as np
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

nltk.download('punkt', quiet=True)


# ── Task 2: Preprocessing Function ──────────────────────────
def preprocess(text: str) -> str:
    """Lowercase and strip extra whitespace."""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# ── Task 3: Define Chatbot Intents ──────────────────────────
intents = {
    "greeting": {
        "patterns": [
            "hello", "hi", "hey", "good morning", "good evening",
            "howdy", "what's up", "hi there", "greetings", "hey there"
        ],
        "responses": [
            "Hello! Welcome to QuickBite Food Delivery. How can I help you today?",
            "Hi there! Ready to order some delicious food?",
            "Hey! Great to see you. What can I get started for you?",
            "Hello! I'm your food delivery assistant. What would you like today?"
        ]
    },
    "order_food": {
        "patterns": [
            "I want to order food", "can I place an order", "I'd like to order",
            "order pizza", "order burger", "I want to buy food",
            "place my order", "I want to get food", "can you take my order",
            "I want to order something", "add to cart", "order now"
        ],
        "responses": [
            "Sure! Please check our menu and tell me what you'd like to order.",
            "Great choice! What would you like to order today?",
            "I'd be happy to take your order. What are you craving?",
            "Let's get your order started! Browse the menu and let me know your items."
        ]
    },
    "menu": {
        "patterns": [
            "what's on the menu", "show me the menu", "what food do you have",
            "what can I order", "list of dishes", "what's available",
            "do you have pizza", "do you serve burgers", "what do you offer",
            "show food items", "what are today's specials", "what cuisines"
        ],
        "responses": [
            "Our menu includes: 🍕 Pizza, 🍔 Burgers, 🌮 Tacos, 🍜 Noodles, 🍗 Fried Chicken, and more! Type 'order food' to place your order.",
            "We offer a wide variety: Pizzas, Burgers, Pasta, Biryani, Sushi, Salads and Desserts. What catches your eye?",
            "Today's highlights: Pepperoni Pizza, Zinger Burger, Chicken Biryani, and Fresh Fruit Salad. Shall I add any of these?",
        ]
    },
    "delivery_time": {
        "patterns": [
            "how long will delivery take", "when will my order arrive",
            "estimated delivery time", "how fast is delivery",
            "what is the delivery time", "delivery eta",
            "how many minutes", "when will food arrive",
            "track my order", "where is my order", "order status"
        ],
        "responses": [
            "Your order will arrive in approximately 30–45 minutes depending on your location.",
            "Our average delivery time is 35 minutes. You'll receive an SMS once the rider is on the way!",
            "Delivery usually takes 25–50 minutes. You can track your order in real time via our app.",
        ]
    },
    "payment": {
        "patterns": [
            "how can I pay", "payment methods", "do you accept credit card",
            "can I pay cash on delivery", "online payment", "debit card",
            "is there a payment option", "how to pay", "do you take easypaisa",
            "JazzCash payment", "COD available", "pay online"
        ],
        "responses": [
            "We accept Cash on Delivery, Credit/Debit Cards, JazzCash, and EasyPaisa!",
            "Payment options: 💵 Cash on Delivery | 💳 Card | 📱 JazzCash | 📱 EasyPaisa. Choose what suits you!",
            "You can pay via card, mobile wallet (JazzCash / EasyPaisa), or simply cash when the order arrives.",
        ]
    },
    "contact": {
        "patterns": [
            "how can I contact you", "customer support", "help",
            "call support", "support number", "complaint",
            "I have a problem", "speak to agent", "contact us",
            "helpline", "live chat", "email support", "reach you"
        ],
        "responses": [
            "You can reach us at 📞 0800-QUICKBITE or email support@quickbite.pk. We're available 24/7!",
            "Need help? Call our helpline: 042-111-222-333 or chat with a live agent on our website.",
            "Our support team is here for you! Email: help@quickbite.pk | Phone: 0300-1234567",
        ]
    }
}


# ── Task 4: Prepare Training Data ────────────────────────────
training_patterns = []
training_intents  = []

for intent_name, intent_data in intents.items():
    for pattern in intent_data["patterns"]:
        training_patterns.append(preprocess(pattern))
        training_intents.append(intent_name)

print(f"✅  Training samples loaded: {len(training_patterns)}")


# ── Task 5 + 6: TF-IDF + Naive Bayes Pipeline ───────────────
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2),   # unigrams + bigrams
                              min_df=1,
                              sublinear_tf=True)),
    ("clf",   MultinomialNB(alpha=0.5))
])

model.fit(training_patterns, training_intents)
print("✅  Model trained successfully!\n")


# ── Task 7: Prediction Function ──────────────────────────────
def predict_intent(user_input: str):
    """Return (predicted_intent, confidence_score)."""
    cleaned = preprocess(user_input)
    proba   = model.predict_proba([cleaned])[0]
    classes = model.classes_
    idx     = np.argmax(proba)
    return classes[idx], round(float(proba[idx]), 4)


# ── Task 8: Response Function ────────────────────────────────
CONFIDENCE_THRESHOLD = 0.25

def get_response(user_input: str) -> str:
    """Predict intent and return an appropriate response."""
    intent, confidence = predict_intent(user_input)

    if confidence < CONFIDENCE_THRESHOLD:
        return ("I'm not quite sure I understood that. 😕  "
                "You can ask me about our menu, placing an order, "
                "delivery time, payment options, or contact details!")

    response = random.choice(intents[intent]["responses"])
    return f"{response}  (Intent: {intent} | Confidence: {confidence:.0%})"


# ── Task 9: Chatbot Loop ─────────────────────────────────────
def run_chatbot():
    print("=" * 55)
    print("  🍔  Welcome to QuickBite Food Delivery Chatbot!  🍕")
    print("  Type 'quit' to exit.")
    print("=" * 55)

    while True:
        user_input = input("\nYou: ").strip()

        if not user_input:
            continue

        if user_input.lower() in {"quit", "exit", "bye"}:
            print("Bot: Thank you for using QuickBite! Have a great meal! 👋")
            break

        print(f"Bot: {get_response(user_input)}")


# ── Entry Point ──────────────────────────────────────────────
if __name__ == "__main__":
    # Quick smoke-test before interactive loop
    test_inputs = [
        "hello there",
        "I want to order a pizza",
        "what's on the menu",
        "how long does delivery take",
        "can I pay with JazzCash",
        "I need help"
    ]
    print("── Quick Test ──────────────────────────────────")
    for t in test_inputs:
        intent, conf = predict_intent(t)
        print(f"  Input : '{t}'")
        print(f"  Intent: {intent}  |  Confidence: {conf:.0%}\n")

    print("── Starting Interactive Chatbot ────────────────")
    run_chatbot()

✅  Training samples loaded: 70
✅  Model trained successfully!

── Quick Test ──────────────────────────────────
  Input : 'hello there'
  Intent: greeting  |  Confidence: 46%

  Input : 'I want to order a pizza'
  Intent: order_food  |  Confidence: 80%

  Input : 'what's on the menu'
  Intent: menu  |  Confidence: 60%

  Input : 'how long does delivery take'
  Intent: delivery_time  |  Confidence: 45%

  Input : 'can I pay with JazzCash'
  Intent: payment  |  Confidence: 56%

  Input : 'I need help'
  Intent: contact  |  Confidence: 41%

── Starting Interactive Chatbot ────────────────
  🍔  Welcome to QuickBite Food Delivery Chatbot!  🍕
  Type 'quit' to exit.
Bot: Hello! I'm your food delivery assistant. What would you like today?  (Intent: greeting | Confidence: 46%)
Bot: I'd be happy to take your order. What are you craving?  (Intent: order_food | Confidence: 80%)
Bot: Our menu includes: 🍕 Pizza, 🍔 Burgers, 🌮 Tacos, 🍜 Noodles, 🍗 Fried Chicken, and more! Type 'order food' to place you